# S4_01 — AndinaLog Productos: Diagnóstico Bronze

**Objetivo:** diagnosticar la calidad del archivo `andinalog_productos.csv` sin corregir el Bronze.

Este Notebook 1:
- carga el CSV Bronze;
- valida el contrato mínimo de columnas;
- perfila estructura, nulos, duplicados y categorías;
- aplica reglas de diagnóstico;
- agrega `fila_bronze`, `columnas_con_problemas` y `en_cuarentena`;
- genera salidas de diagnóstico para el siguiente paso del pipeline.

> **Importante:** aquí no se corrigen datos. Las correcciones/tratamientos corresponden al Notebook 2.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

# Si trabajas en Google Colab con Drive, cambia BASE_DIR por tu carpeta.
BASE_DIR = Path(".")
BRONZE_FILE = BASE_DIR / "andinalog_productos.csv"

OUTPUT_DIAGNOSTICADO = BASE_DIR / "andinalog_productos_diagnosticado.csv"
OUTPUT_PROBLEMAS = BASE_DIR / "andinalog_productos_problemas.csv"
OUTPUT_CUARENTENA = BASE_DIR / "andinalog_productos_cuarentena.csv"
OUTPUT_REPORTE = BASE_DIR / "andinalog_productos_reporte_calidad.csv"

print("Bronze:", BRONZE_FILE)


## 1. Carga y contrato del Bronze

In [ ]:
df = pd.read_csv(BRONZE_FILE)

print(f"Filas: {len(df):,}")
print(f"Columnas: {len(df.columns)}")
display(df.head())


In [ ]:
COLUMNAS_ESPERADAS = [
    "producto_id",
    "nombre_producto",
    "categoria_logistica",
    "temperatura_conservacion_requerida_c",
    "tolerancia_temperatura_c",
    "precio_unitario_bob",
    "costo_unitario_bob",
]

faltantes = [c for c in COLUMNAS_ESPERADAS if c not in df.columns]
extras = [c for c in df.columns if c not in COLUMNAS_ESPERADAS]

print("Columnas faltantes:", faltantes)
print("Columnas extra:", extras)

if faltantes:
    raise ValueError(f"El Bronze no cumple el contrato mínimo. Faltan: {faltantes}")


## 2. Perfilado inicial

In [ ]:
perfil = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "porcentaje_nulos": (df.isna().mean() * 100).round(2),
    "unicos": df.nunique(dropna=True),
})
display(perfil)


In [ ]:
print("Duplicados exactos:", df.duplicated().sum())
print("IDs duplicados:", df.duplicated(subset=["producto_id"], keep=False).sum())

print("\nCategorías observadas:")
display(df["categoria_logistica"].value_counts(dropna=False).to_frame("filas"))

print("\nCombinaciones categoría / temperatura / tolerancia:")
display(
    df.groupby(
        ["categoria_logistica",
         "temperatura_conservacion_requerida_c",
         "tolerancia_temperatura_c"],
        dropna=False
    ).size().reset_index(name="filas")
)


## 3. Catálogo de reglas de diagnóstico

Las reglas se aplican sobre el Bronze **sin modificar sus valores**.

- **P01** — campos obligatorios nulos.
- **P02** — formato de `producto_id` distinto de `PROD-999`.
- **P03** — `producto_id` duplicado.
- **P04** — categoría logística fuera del catálogo permitido.
- **P05** — temperatura/tolerancia incompatible con la categoría.
- **P06** — precio o costo no positivo.
- **P07** — costo unitario mayor que el precio unitario.

Para la coherencia térmica se usa el patrón observado/esperado del dominio del dataset:
`Seco → 20°C ±5`, `Fresco → 4°C ±2`, `Congelado → -18°C ±2`.


In [ ]:
REGLAS = {
    "P01": "Campo obligatorio nulo",
    "P02": "Formato de producto_id inválido",
    "P03": "producto_id duplicado",
    "P04": "Categoría logística no válida",
    "P05": "Temperatura/tolerancia incompatible con la categoría",
    "P06": "Precio o costo no positivo",
    "P07": "Costo unitario mayor que precio unitario",
}

CATEGORIAS_VALIDAS = {"Seco", "Fresco", "Congelado"}

CONDICIONES_TERMICAS = {
    "Seco": (20.0, 5.0),
    "Fresco": (4.0, 2.0),
    "Congelado": (-18.0, 2.0),
}

pd.DataFrame(
    [{"regla": k, "descripcion": v} for k, v in REGLAS.items()]
)


## 4. Aplicación de reglas y cuarentena

In [ ]:
diagnostico = df.copy()

# Número de fila del archivo Bronze considerando cabecera en la fila 1.
diagnostico.insert(0, "fila_bronze", np.arange(2, len(diagnostico) + 2))

problemas = []

def registrar(mask, regla, columnas):
    indices = diagnostico.index[mask]
    for idx in indices:
        problemas.append({
            "fila_bronze": int(diagnostico.at[idx, "fila_bronze"]),
            "producto_id": diagnostico.at[idx, "producto_id"],
            "regla": regla,
            "descripcion": REGLAS[regla],
            "columnas": ", ".join(columnas),
        })

# P01: obligatorios
obligatorias = COLUMNAS_ESPERADAS
for col in obligatorias:
    registrar(diagnostico[col].isna(), "P01", [col])

# P02: formato ID exacto
mask = ~diagnostico["producto_id"].astype("string").str.match(r"^PROD-\d{3}$", na=False)
registrar(mask, "P02", ["producto_id"])

# P03: duplicidad de ID tal como aparece en Bronze
mask = diagnostico["producto_id"].duplicated(keep=False)
registrar(mask, "P03", ["producto_id"])

# P04: categorías exactas permitidas
mask = ~diagnostico["categoria_logistica"].isin(CATEGORIAS_VALIDAS)
registrar(mask, "P04", ["categoria_logistica"])

# P05: coherencia térmica solo para categorías reconocidas y valores presentes
for categoria, (temp, tol) in CONDICIONES_TERMICAS.items():
    mask_cat = diagnostico["categoria_logistica"].eq(categoria)
    mask_valores = (
        diagnostico["temperatura_conservacion_requerida_c"].notna()
        & diagnostico["tolerancia_temperatura_c"].notna()
    )
    mask_inconsistente = mask_cat & mask_valores & (
        ~np.isclose(diagnostico["temperatura_conservacion_requerida_c"], temp)
        | ~np.isclose(diagnostico["tolerancia_temperatura_c"], tol)
    )
    registrar(
        mask_inconsistente,
        "P05",
        ["categoria_logistica",
         "temperatura_conservacion_requerida_c",
         "tolerancia_temperatura_c"],
    )

# P06
mask = (diagnostico["precio_unitario_bob"] <= 0) | (diagnostico["costo_unitario_bob"] <= 0)
registrar(mask, "P06", ["precio_unitario_bob", "costo_unitario_bob"])

# P07
mask = diagnostico["costo_unitario_bob"] > diagnostico["precio_unitario_bob"]
registrar(mask, "P07", ["precio_unitario_bob", "costo_unitario_bob"])

problemas_df = pd.DataFrame(
    problemas,
    columns=["fila_bronze", "producto_id", "regla", "descripcion", "columnas"]
)

display(problemas_df.head(20))
print("Problemas detectados:", len(problemas_df))


In [ ]:
# Consolidar columnas problemáticas por fila
if len(problemas_df):
    columnas_por_fila = (
        problemas_df.groupby("fila_bronze")["columnas"]
        .apply(lambda s: sorted({
            col.strip()
            for texto in s
            for col in texto.split(",")
            if col.strip()
        }))
        .to_dict()
    )
else:
    columnas_por_fila = {}

diagnostico["columnas_con_problemas"] = diagnostico["fila_bronze"].map(
    lambda fila: ", ".join(columnas_por_fila.get(fila, []))
)
diagnostico["en_cuarentena"] = diagnostico["columnas_con_problemas"].ne("")

cuarentena = diagnostico[diagnostico["en_cuarentena"]].copy()

print(f"Filas Bronze: {len(diagnostico):,}")
print(f"Filas en cuarentena: {len(cuarentena):,}")
print(f"Filas sin problemas detectados: {(~diagnostico['en_cuarentena']).sum():,}")
display(cuarentena)


## 5. Reporte de calidad

In [ ]:
if len(problemas_df):
    reporte_reglas = (
        problemas_df.groupby(["regla", "descripcion"], as_index=False)
        .agg(ocurrencias=("fila_bronze", "size"),
             filas_afectadas=("fila_bronze", "nunique"))
    )
else:
    reporte_reglas = pd.DataFrame(
        columns=["regla", "descripcion", "ocurrencias", "filas_afectadas"]
    )

display(reporte_reglas)

resumen = pd.DataFrame([{
    "filas_bronze": len(diagnostico),
    "filas_utilizables_sin_tratamiento": int((~diagnostico["en_cuarentena"]).sum()),
    "filas_en_cuarentena": int(diagnostico["en_cuarentena"].sum()),
    "porcentaje_cuarentena": round(diagnostico["en_cuarentena"].mean() * 100, 2),
    "problemas_detectados": len(problemas_df),
}])

display(resumen)


## 6. Exportación reproducible

In [ ]:
diagnostico.to_csv(OUTPUT_DIAGNOSTICADO, index=False)
problemas_df.to_csv(OUTPUT_PROBLEMAS, index=False)
cuarentena.to_csv(OUTPUT_CUARENTENA, index=False)
reporte_reglas.to_csv(OUTPUT_REPORTE, index=False)

print("Generados:")
for archivo in [
    OUTPUT_DIAGNOSTICADO,
    OUTPUT_PROBLEMAS,
    OUTPUT_CUARENTENA,
    OUTPUT_REPORTE,
]:
    print(" -", archivo)


## Resultado del Notebook 1

Este notebook deja el dataset **diagnosticado**, pero conserva intactos los valores originales del Bronze.

El siguiente paso es el **Notebook 2**, donde se define el tratamiento de cada problema aceptado por el catálogo de reglas y se genera el **Silver de Productos**.
